# Time-dependent transition rates

This notebook shows how to use `SimulationState` to make latent transition rates depend on simulation time or on the current cell population.

*Note: `markovmodus` uses a fixed-`dt` discretized CTMC approximation. Dynamic transition rates are evaluated at the start of each time step and held constant over that step.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from markovmodus import SimulationParameters, SimulationState, simulate_dataset

In [ ]:
def state_distribution(adata, label):
    counts = adata.obs["state"].value_counts().sort_index()
    return pd.DataFrame({"scenario": label, "state": counts.index, "cells": counts.values})


base_kwargs = dict(
    num_states=3,
    num_genes=150,
    num_cells=1000,
    t_final=20.0,
    dt=0.5,
    markers_per_state=50,
    initial_state_probs=[1.0, 0.0, 0.0],
    rng_seed=12,
)

## Examples

### 1. Static baseline

A static transition matrix is still the simplest entry point. Rows are source states and columns are destination states; `transition_rates` receives the same numeric matrix object.

In [ ]:
transition_matrix = np.array(
    [
        [0.0, 0.05, 0.0],
        [0.0, 0.0, 0.05],
        [0.0, 0.0, 0.0],
    ],
    dtype=float,
)

static_params = SimulationParameters(**base_kwargs, transition_rates=transition_matrix)
static_adata = simulate_dataset(static_params)

### 2. Time-dependent rates

To generalize to time-dependent transition rates, return a transition matrix from a function, and let individual entries depend on `state.time`. In the example, the transition 0 -> 1 jumps after time 8, while 1 -> 2 turns on following a sigmoid.

In [ ]:
def sigmoid(x: float, *, midpoint: float, scale: float) -> float:
    return 1.0 / (1.0 + np.exp(-(x - midpoint) / scale))


def time_dependent_rates(state: SimulationState) -> np.ndarray:
    transition_matrix = np.zeros((3, 3), dtype=float)
    transition_matrix[0, 1] = 0.02 if state.time < 8.0 else 0.18
    transition_matrix[1, 2] = 0.02 + 0.16 * sigmoid(state.time, midpoint=12.0, scale=1.5)
    return transition_matrix


time_params = SimulationParameters(**base_kwargs, transition_rates=time_dependent_rates)
time_adata = simulate_dataset(time_params)

### 3. Population-dependent rates

The same pattern works for population feedback: set up a transition rate that depends on `state.cell_states`, then use it in a matrix entry. Here the 1 -> 2 rate is damped by a Hill-like term using the number of cells already in the third state.

In [ ]:
def population_feedback_rates(state: SimulationState) -> np.ndarray:
    counts = np.bincount(state.cell_states, minlength=3)
    number_of_cells_in_state_3 = counts[2]
    base_transition = 0.12

    transition_matrix = np.zeros((3, 3), dtype=float)
    transition_matrix[0, 1] = 0.05
    transition_matrix[1, 2] = base_transition / (1.0 + number_of_cells_in_state_3)
    return transition_matrix


feedback_params = SimulationParameters(**base_kwargs, transition_rates=population_feedback_rates)
feedback_adata = simulate_dataset(feedback_params)

## Comparison of the three cases

In [ ]:
comparison = pd.concat(
    [
        state_distribution(static_adata, "static"),
        state_distribution(time_adata, "time-dependent"),
        state_distribution(feedback_adata, "population feedback"),
    ],
    ignore_index=True,
)

comparison.pivot(index="state", columns="scenario", values="cells").fillna(0).astype(int)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for scenario, group in comparison.groupby("scenario"):
    ax.bar(
        group["state"] + 0.22 * list(comparison["scenario"].unique()).index(scenario),
        group["cells"],
        width=0.2,
        label=scenario,
    )

ax.set_xlabel("final latent state")
ax.set_ylabel("cells")
ax.legend()
plt.tight_layout()

The next cells use Scanpy to project each simulated dataset. Install the optional dependency with `pip install scanpy` if needed.

In [ ]:
import scanpy as sc

sc.settings.set_figure_params(dpi=100)


def preprocess_for_umap(adata, *, n_hvg: int = 100):
    adata = adata.copy()
    adata.obs["state_label"] = [f"state_{state}" for state in adata.obs["state"]]
    adata.obs["state_label"] = adata.obs["state_label"].astype("category")

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvg)
    adata = adata[:, adata.var["highly_variable"]].copy()
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=30)
    sc.pp.neighbors(adata, n_neighbors=15)
    sc.tl.umap(adata)
    return adata

In [ ]:
static_processed = preprocess_for_umap(static_adata)
time_processed = preprocess_for_umap(time_adata)
feedback_processed = preprocess_for_umap(feedback_adata)

umap_rows = [
    ("Static rates", static_processed),
    ("Time-dependent rates", time_processed),
    ("Population feedback", feedback_processed),
]

fig, axes = plt.subplots(1, len(umap_rows), figsize=(15, 4.5))

for ax, (title, processed) in zip(axes, umap_rows):
    sc.pl.umap(
        processed,
        color="state_label",
        show=False,
        ax=ax,
        legend_loc="right margin",
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")

fig.tight_layout()